In [9]:
import os
import requests
import re
from bs4 import BeautifulSoup


# ---------------- FETCH PAGE ----------------
def get_page():
    global url

    try:
        url = input("Enter url of a Medium article: ").strip()

        #  validate Medium URL only
        if not re.match(r'https?://medium\.com/', url):
            raise ValueError("Please enter a valid Medium article URL")

        #  convert to Jina reader URL
        jina_url = "https://r.jina.ai/" + url

        headers = {
            "User-Agent": "Mozilla/5.0",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }

        session = requests.Session()
        response = session.get(jina_url, headers=headers, timeout=15)
        response.raise_for_status()

        return BeautifulSoup(response.text, "html.parser")

    except ValueError as ve:
        print("Input Error:", ve)

    except requests.exceptions.Timeout:
        print("Request timed out. Try again.")

    except requests.exceptions.RequestException as e:
        print("Network/HTTP Error:", e)

    except Exception as e:
        print("Unexpected error:", e)

    return None


# ---------------- EXTRACT TEXT (ROBUST) ----------------
def collect_text(soup):
    if soup is None:
        return "No data scraped due to earlier error."

    output = f"url: {url}\n\n"

    # Try structured extraction first
    tags = soup.find_all(["h1", "h2", "h3", "p"])

    content = []

    for tag in tags:
        text = tag.get_text(strip=True)
        if text:
            content.append(text)

    # Fallback if structured parsing fails
    if not content:
        raw_text = soup.get_text(separator="\n", strip=True)

        if raw_text:
            return output + raw_text
        else:
            return output + "No readable content found."

    # Clean join
    return output + "\n\n".join(content)


# ---------------- SAVE FILE ----------------
def save_file(text):
    try:
        folder = "scraped_articles"

        if not os.path.exists(folder):
            os.makedirs(folder)

        # safe filename
        name = url.split("/")[-1]
        file_path = os.path.join(folder, f"{name}.txt")

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text)

        print(f"\nFile saved at: {file_path}")

    except Exception as e:
        print("Error while saving file:", e)


# ---------------- MAIN ----------------
if __name__ == "__main__":
    soup = get_page()
    text = collect_text(soup)
    save_file(text)

Enter url of a Medium article:  https://medium.com/@subashgandyer/papa-what-is-a-neural-network-c5e5cc427c7



File saved at: scraped_articles\papa-what-is-a-neural-network-c5e5cc427c7.txt
